# Projeto Python IA: Inteligência Artificial e Previsões

### Case: Score de Crédito dos Clientes

Você foi contratado por um banco para conseguir definir o score de crédito dos clientes. Você precisa analisar todos os clientes do banco e, com base nessa análise, criar um modelo que consiga ler as informações do cliente e dizer automaticamente o score de crédito dele: Ruim, Ok, Bom

Arquivos da aula: https://drive.google.com/drive/folders/1FbDqVq4XLvU85VBlVIMJ73p9oOu6u2-J?usp=drive_link

In [ ]:
# ==========================
# PROJETO IA - VERIFICAÇÃO DE FARDOS
# ==========================

import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor

# ==========================
# 1) Leitura da base de dados
# ==========================
tabela = pd.read_excel("Banco_de_Fardos.xlsx")
tabela.columns = tabela.columns.str.strip()

# Garantir que "Fator" seja numérico
tabela["Fator"] = pd.to_numeric(tabela["Fator"], errors="coerce").fillna(0)

# Garantir que "Conversao" esteja como string
tabela["Conversao"] = tabela["Conversao"].astype(str)

print("\nTipos das colunas iniciais:")
print(tabela.dtypes)

# ==========================
# 2) Codificação das colunas categóricas
# ==========================
codificadores = {}
for coluna in ["EAN", "Material", "Nome_CONC"]:  # ajuste os nomes conforme sua planilha
    le = LabelEncoder()
    
    # inclui "Desconhecido" na lista de classes
    valores = tabela[coluna].astype(str).fillna("Desconhecido").values
    le.fit(np.append(valores, "Desconhecido"))
    
    # transforma os dados
    tabela[coluna] = le.transform(valores)
    
    codificadores[coluna] = le

# ==========================
# 3) Separar features e alvos
# ==========================
X = tabela.drop(columns=["Conversao", "Fator"])
y1 = tabela["Conversao"]  # classificação
y2 = tabela["Fator"]      # regressão

# ==========================
# 4) Criar e treinar os modelos (100% treino)
# ==========================
modelo_conversao = RandomForestClassifier(random_state=42)
modelo_fator = RandomForestRegressor(random_state=42)

modelo_conversao.fit(X, y1)
modelo_fator.fit(X, y2)

# ==========================
# 5) Previsão em novos fardos
# ==========================
novos_fardos = pd.read_excel("Novos_Fardos.xlsx")
novos_fardos.columns = novos_fardos.columns.str.strip()

for coluna, codificador in codificadores.items():
    if coluna in novos_fardos.columns:
        valores = novos_fardos[coluna].astype(str).fillna("Desconhecido")
        novos_fardos[coluna] = [
            codificador.transform([v])[0] if v in codificador.classes_ else codificador.transform(["Desconhecido"])[0]
            for v in valores
        ]

# ==========================
# 6) Fazer previsões
# ==========================
X_novos = novos_fardos[X.columns]

previsao_conversao = modelo_conversao.predict(X_novos)
previsao_fator = modelo_fator.predict(X_novos)

# ==========================
# 7) Montar resultados finais
# ==========================
resultado = novos_fardos.copy()
resultado["Conversao"] = previsao_conversao
resultado["Fator"] = previsao_fator

# Decodificar de volta as colunas categóricas
for coluna, codificador in codificadores.items():
    if coluna in resultado.columns:
        resultado[coluna] = codificador.inverse_transform(
            resultado[coluna].astype(int).to_numpy()
        )

# ==========================
# 8) Exportar para Excel
# ==========================
saida = r"C:\Users\max.sousa\Downloads\PythonTestes\ProjetoIA\PlanilhaAtualizada.xlsx"
resultado.to_excel(saida, index=False)

print(f"\n✅ Resultado salvo em: {saida}")
